# 3: BERTopic Topic Merging & Counts

**Newest version**: 23-04-2026

Merges the two hierarchical topic models produced in `2_BERTopic topic modelling`:
- **Main model**: trained on all ~5.3 M documents
- **Outlier model**: trained only on the ~1.2 M documents the main model left as topic `-1`

**Sections**
1. Imports & paths
2. Load data & models
3. Pre-merge checks
4. Correct topic counts
5. Merge topic assignments
6. Merged topic info & counts
7. Rename topic labels (human coding)
8. Add overarching categories (human coding)
9. Find fragmented / overlapping topics
10. Link topics to videos
11. Save outputs

---

## 1 · Imports & paths

In [ ]:
import pickle
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from bertopic import BERTopic
from config import *

## 2 · Load data & models

In [ ]:
# ── Raw hashtag data ─────────────────────────────────────────────────────────
hashtag_df = pd.read_pickle(HASHTAG_DF_PATH)

# ── Documents ────────────────────────────────────────────────────────────────
with open(FINAL_DOCS_PATH, "rb") as f:
    docs_bert = pickle.load(f)

with open(OUTLIER_DOCS_PATH, "rb") as f:
    outlier_docs = pickle.load(f)

# ── Topic models ─────────────────────────────────────────────────────────────
topic_model          = BERTopic.load(TOPIC_MODEL_PATH)
topic_model_outliers = BERTopic.load(OUTLIER_MODEL_PATH)

# ── Topic assignments ────────────────────────────────────────────────────────
topics          = np.load(TOPICS_ASSIGNED_PATH)
topics_outliers = np.load(OUTLIER_TOPICS_PATH)

print("✓ All data loaded")
print(f"  Videos:            {len(hashtag_df):,}")
print(f"  Documents:         {len(docs_bert):,}")
print(f"  Main topics:       {len(topics):,}")
print(f"  Outlier docs:      {len(outlier_docs):,}")
print(f"  Outlier topics:    {len(topics_outliers):,}")

✓ All data loaded
  Videos:            5,332,704
  Documents:         5,332,704
  Main topics:       5,332,704
  Outlier docs:      1,541,249
  Outlier topics:    1,541,249


In [ ]:
hashtag_df.head()

## 3 · Pre-merge checks

Verify that the outlier model was actually run on the outliers from the main model —  
i.e. the number of main-model outliers must equal the number of outlier-model documents.

In [7]:
outliers_in_main       = (topics == -1).sum()
docs_in_outlier_model  = len(topics_outliers)
match                  = outliers_in_main == docs_in_outlier_model

print("-" * 60)
print("PRE-MERGE VERIFICATION")
print("-" * 60)
print(f"Main model:")
print(f"  Total docs:      {len(topics):,}")
print(f"  Assigned:        {(topics != -1).sum():,}")
print(f"  Outliers (-1):   {outliers_in_main:,}")
print(f"\nOutlier model:")
print(f"  Total docs:      {docs_in_outlier_model:,}")
print(f"  Assigned:        {(topics_outliers != -1).sum():,}")
print(f"  Still outliers:  {(topics_outliers == -1).sum():,}")
print(f"\nCounts match: {'✅' if match else '❌  WARNING — models do not belong together!'}")

------------------------------------------------------------
PRE-MERGE VERIFICATION
------------------------------------------------------------
Main model:
  Total docs:      5,332,704
  Assigned:        3,791,455
  Outliers (-1):   1,541,249

Outlier model:
  Total docs:      1,541,249
  Assigned:        971,961
  Still outliers:  569,288

Counts match: ✅


## 4 · Correct topic counts

`.get_topic_info()` reports counts that only sum to the batch size used during training,  
not the full corpus. The function below recalculates true counts from the actual assignment arrays.

In [8]:
# Replace the model-reported counts with true counts computed from the full topic assignment array.
def get_corrected_topic_info(topic_model: BERTopic, topics: np.ndarray) -> pd.DataFrame:
    base_info   = topic_model.get_topic_info()
    true_counts = Counter(topics)
    base_info["Count"] = base_info["Topic"].map(true_counts).fillna(0).astype(int)
    return base_info.sort_values("Count", ascending=False).reset_index(drop=True)


def _print_count_verification(label: str, topic_model: BERTopic,
                              corrected_info: pd.DataFrame, topics: np.ndarray) -> None:
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"  Topics (excl. -1):      {len(corrected_info[corrected_info['Topic'] != -1])}")
    print(f"  Total documents:        {corrected_info['Count'].sum():,}")
    print(f"  Model-reported (wrong): {topic_model.get_topic_info()['Count'].sum():,}")
    print(f"  Actual assignments:     {len(topics):,}")
    ok = corrected_info["Count"].sum() == len(topics)
    print(f"  Counts match:           {'✅' if ok else '❌'}")

In [9]:
corrected_topic_info   = get_corrected_topic_info(topic_model,          topics)
corrected_outlier_info = get_corrected_topic_info(topic_model_outliers, topics_outliers)

_print_count_verification("MAIN MODEL",    topic_model,          corrected_topic_info,   topics)
_print_count_verification("OUTLIER MODEL", topic_model_outliers, corrected_outlier_info, topics_outliers)


MAIN MODEL
  Topics (excl. -1):      228
  Total documents:        5,332,704
  Model-reported (wrong): 50,000
  Actual assignments:     5,332,704
  Counts match:           ✅

OUTLIER MODEL
  Topics (excl. -1):      248
  Total documents:        1,541,249
  Model-reported (wrong): 50,000
  Actual assignments:     1,541,249
  Counts match:           ✅


In [10]:
corrected_topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1541249,-1_meme_anime_funny_bnha,"[meme, anime, funny, bnha, mha, greenscreen, c...","[yougetmeyuh ehenn carbon hayeckpuur, memes😂 😂..."
1,6,154565,6_the100_julieandthephantoms_charlidamelio_reign,"[the100, julieandthephantoms, charlidamelio, r...",[reign queenvictoria maryqueenotscots reignedi...
2,7,124934,7_fitness_weightloss_workout_gym,"[fitness, weightloss, workout, gym, caloriedef...","[weightloss gym workout christmas2023 fitness,..."
3,0,103160,0_food_recipe_cooking_tiktokfood,"[food, recipe, cooking, tiktokfood, baking, ea...",[eat yummy quickrecipes easyrecipe cooking tik...
4,17,95411,17_netflix_movie_film_outerbanks,"[netflix, movie, film, outerbanks, winx, winxc...","[netflix, netflix herder sarahgrey theordernet..."


In [11]:
corrected_outlier_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,569288,-1_music_cute_fashion_funny,"[music, cute, fashion, funny, anime, love, rel...",[alttiktok stitch kuromi cosplay cute hellokit...
1,1,27308,1_taylorswift_swifttok_erastour_swiftie,"[taylorswift, swifttok, erastour, swiftie, tay...",[wizzair erastour swiftie taylorswift swifttok...
2,0,24020,0_football_soccer_footballtiktok_futbol,"[football, soccer, footballtiktok, futbol, pre...","[baggio delpiero totti football, philfoden foo..."
3,2,20749,2_comedy_humor_funny_funnyvideos,"[comedy, humor, funny, funnyvideos, comedia, l...","[funnyvideos comedy funny jimmycarr ruthless, ..."
4,3,19419,3_genshinimpact_genshin_scaramouche_genshinmemes,"[genshinimpact, genshin, scaramouche, genshinm...","[genshin genshinimpact, ninnguanggenshin ningg..."


In [ ]:
# Uncomment to save
# corrected_topic_info.to_excel(CORRECTED_TOPIC_INFO_PATH, index=False)
# corrected_outlier_info.to_excel(CORRECTED_OUTLIER_INFO_PATH, index=False)

## 5 · Merge topic assignments

Each model assigns topic IDs independently starting from 0, so IDs from the two models  
would clash. The merge adds an **offset** equal to `max(main_topic_id) + 1` to all  
outlier-model IDs before splicing them into the main assignment array.

In [12]:
# Combine main-model and outlier-model topic assignments into a single array
def merge_topic_assignments(
    topics_main: np.ndarray,
    topics_outliers: np.ndarray,
) -> tuple[np.ndarray, dict]:
    """
    Returns
    merged_topics    : Combined assignment array (same length as topics_main).
    topic_id_mapping : Metadata dict with offset and ID ranges for reference.
    """
    max_main  = topics_main[topics_main != -1].max()
    offset    = int(max_main) + 1

    merged    = topics_main.copy()
    out_off   = topics_outliers.copy()
    mask      = out_off != -1
    out_off[mask] += offset

    outlier_indices = np.where(topics_main == -1)[0]

    if len(outlier_indices) != len(out_off):
        raise ValueError(
            f"Length mismatch: {len(outlier_indices):,} main outliers "
            f"vs {len(out_off):,} outlier-model assignments."
        )

    merged[outlier_indices] = out_off

    max_out = out_off[mask].max() if mask.any() else offset
    mapping = {
        "offset":                  offset,
        "main_topic_range":        (0, max_main),
        "outlier_topic_range_raw": (0, max_out - offset),
        "outlier_topic_range_merged": (offset, max_out),
    }

    print("-" * 60)
    print("MERGE SUMMARY")
    print("-" * 60)
    print(f"  Main topic IDs:         0 – {max_main}")
    print(f"  Outlier ID offset:      {offset}")
    print(f"  Outlier topic IDs:      {offset} – {max_out}")
    print(f"  Total documents:        {len(merged):,}")
    print(f"  Unique topics (main):   {len(np.unique(topics_main[topics_main != -1]))}")
    print(f"  Unique topics (outlier):{len(np.unique(topics_outliers[topics_outliers != -1]))}")
    print(f"  Unique topics (merged): {len(np.unique(merged[merged != -1]))}")
    n_out = (merged == -1).sum()
    print(f"  Final outliers:         {n_out:,}  ({n_out/len(merged)*100:.2f}%)")

    return merged, mapping

In [13]:
merged_topics, topic_id_mapping = merge_topic_assignments(
    topics_main=topics,
    topics_outliers=topics_outliers,
)

------------------------------------------------------------
MERGE SUMMARY
------------------------------------------------------------
  Main topic IDs:         0 – 227
  Outlier ID offset:      228
  Outlier topic IDs:      228 – 475
  Total documents:        5,332,704
  Unique topics (main):   228
  Unique topics (outlier):248
  Unique topics (merged): 476
  Final outliers:         569,288  (10.68%)


In [ ]:
# Uncomment to save
# np.save(MERGED_TOPICS_PATH, merged_topics)

## 6 · Merged topic info & counts

Combine the two topic info DataFrames and recalculate all counts from `merged_topics`.

In [ ]:
def merge_topic_info_with_correct_counts(
    corrected_main_info: pd.DataFrame,
    corrected_outlier_info: pd.DataFrame,
    merged_topics: np.ndarray,
    offset: int,
) -> pd.DataFrame:

    true_counts = Counter(merged_topics)

    main_info    = corrected_main_info.copy()
    outlier_info = corrected_outlier_info.copy()

    # Offset outlier IDs (except -1) so they match merged_topics
    outlier_info.loc[outlier_info["Topic"] != -1, "Topic"] += offset
    # Drop the outlier model's own -1 row; the main model's -1 row is kept
    outlier_info = outlier_info[outlier_info["Topic"] != -1]

    main_info["Source"]    = "Main Model"
    outlier_info["Source"] = "Outlier Model"

    combined = pd.concat([main_info, outlier_info], ignore_index=True)
    combined["Count"] = combined["Topic"].map(true_counts).fillna(0).astype(int)
    combined = combined[combined["Count"] > 0]  # drop topics absent from merged array

    outliers_row    = combined[combined["Topic"] == -1]
    non_outliers    = combined[combined["Topic"] != -1].sort_values("Count", ascending=False)
    merged_info     = pd.concat([outliers_row, non_outliers]).reset_index(drop=True)

    print(f"✅ Merged topic info: {len(merged_info)} rows, "
          f"{merged_info['Count'].sum():,} documents total")
    print(f"   Outliers (-1):     {merged_info.loc[merged_info['Topic'] == -1, 'Count'].values[0]:,}")
    print(f"   Topics with 0 docs removed: {(combined['Count'] == 0).sum()}")

    return merged_info

In [16]:
merged_topic_info = merge_topic_info_with_correct_counts(
    corrected_main_info=corrected_topic_info,
    corrected_outlier_info=corrected_outlier_info,
    merged_topics=merged_topics,
    offset=topic_id_mapping["offset"],
)

✅ Merged topic info: 477 rows, 5,332,704 documents total
   Outliers (-1):     569,288
   Topics with 0 docs removed: 0


In [ ]:
merged_topic_info.sample(5, random_state=42)

In [ ]:
# Uncomment to save
# merged_topic_info.to_csv(MERGED_INFO_CSV_PATH, index=False)

### Optional: reload from disk (skip steps above)

Run this cell to jump straight to renaming if the merge has already been saved.

In [ ]:
# merged_topic_info = pd.read_csv(MERGED_INFO_CSV_PATH)
# merged_topics     = np.load(MERGED_TOPICS_PATH, allow_pickle=True)
# print(f"Loaded: {len(merged_topics):,} assignments, {len(merged_topic_info)} topic rows")

## 7 · Rename topic labels (human coding)

Topics were named during qualitative coding sessions, identifying labels, duplicates/overlapping, or incoherent topics.

In [ ]:
def rename_merged_topic_labels(
    merged_topic_info: pd.DataFrame,
    main_topic_mapping: dict,
    outlier_topic_mapping: dict,
    offset: int,
) -> pd.DataFrame:
    """
    Apply human-coded topic labels to the merged topic info DataFrame.
    The original BERTopic name is preserved in 'topic_label_original'.
    """
    df = merged_topic_info.copy()

    # Keep original labels for reference
    df["topic_label_original"] = df["Name"]

    # Create combined mapping (offset outlier topic IDs)
    combined_mapping = dict(main_topic_mapping)  # copy
    for orig_id, label in outlier_topic_mapping.items():
        if orig_id != -1:
            combined_mapping[orig_id + offset] = label 

    df["Name"] = df["Topic"].map(combined_mapping).fillna(df["Name"])

    renamed = (df["Name"] != df["topic_label_original"]).sum()
    print(f"✅ Renamed {renamed} topic labels")
    print(f"   Main mapping entries:    {len(main_topic_mapping)}")
    print(f"   Outlier mapping entries: {len(outlier_topic_mapping)}")
    return df

Below two dictionaries are created where the IDs need to correspond to the original topic IDs from the main model and outlier model, found in:

manual coding/Final topics IDs.xlsx  
manual coding/Incoherent topics IDs.xlsx             

In [18]:
# ── Main model labels ────────────────────────────────────────────────────────
main_topic_mapping = {
    -1: 'Outliers', 0: 'Cooking', 1: 'Art', 2: 'Incoherent', 3: 'Dogs',
    4: 'Harry Potter', 5: 'LGBTQ+', 6: 'TV shows/movies', 7: 'Fitness',
    8: 'Hair', 9: 'School', 10: 'Fashion', 11: 'Netherlands', 12: 'Halloween',
    13: 'Christmas', 14: 'Gaming', 15: 'Funny', 16: 'Disney', 17: 'TV shows/movies',
    18: 'Make-up', 19: 'Incoherent', 20: 'Incoherent', 21: 'Incoherent', 22: 'Marvel',
    23: 'Incoherent', 24: 'Couples', 25: 'Animal Crossing', 26: 'Parenting',
    27: 'Dance', 28: 'Incoherent', 29: 'Anime', 30: 'Football', 31: 'K-Pop',
    32: 'Cats', 33: 'My Hero Academia', 34: 'Witchcraft', 35: 'Skincare', 36: 'COVID-19',
    37: 'Gymnastics', 38: 'The Vampire Diaries', 39: 'Asia', 40: 'Furries',
    41: 'Cars', 42: 'Goth', 43: 'Naruto', 44: 'Avatar: The Last Airbender',
    45: 'The Boyz (Kpop)', 46: 'Friends', 47: 'Basketball', 48: 'My Hero Academia',
    49: 'Dangan Ronpa', 50: 'Photography', 51: 'iPhone', 52: 'Education', 53: 'Incoherent',
    54: 'Fish', 55: 'Incoherent', 56: 'Incoherent', 57: 'Small business', 58: 'Incoherent',
    59: 'Summer', 60: 'Books', 61: 'Singing', 62: 'Incoherent', 63: 'Cosplay',
    64: 'Horses', 65: 'My Hero Academia', 66: 'Funny', 67: 'Storytime', 68: 'Beach', 69: 'Challenges',
    70: 'Animation', 71: 'Gifts', 72: 'Astrology', 73: 'Couples Pranks', 74: 'Quarantine',
    75: 'Languages', 76: 'Europe', 77: 'Incoherent', 78: 'Incoherent', 79: 'Mental Health',
    80: 'Dating', 81: 'Siblings', 82: 'Incoherent', 83: 'Motorcycles', 84: 'NCT', 85: 'Incoherent',
    86: 'Acting', 87: 'Funny', 88: 'Ski/Snowboard', 89: 'Sewing', 90: 'Travel',
    91: 'Commonwealth', 92: 'Sneakers/Shoes', 93: 'Interior Design', 94: 'Incoherent',
    95: 'Guitar', 96: 'Stray Kidz', 97: 'Black Lives Matter', 98: 'Among Us', 99: 'Musical Theatre',
    100: 'Journalling', 101: 'My Hero Academia', 102: 'Skateboarding', 103: 'Tutorials/How to',
    104: 'Mystic Messenger', 105: 'Roblox', 106: 'Bugha', 107: 'My Hero Academia',
    108: 'My Hero Academia', 109: 'Fortnite', 110: 'Military', 111: 'Incoherent',
    112: 'Pranks', 113: 'Snakes', 114: 'Magic Tricks', 115: 'Incoherent', 116: 'Coffee',
    117: 'Music production', 118: 'Minecraft', 119: 'Incoherent', 120: 'Nails', 121: 'Incoherent',
    122: 'Police', 123: 'Hip Hop/Rap', 124: 'Swimming', 125: 'Body modification',
    126: 'Instagram', 127: 'Birds', 128: 'Sad quotes', 129: 'Memes', 130: 'Snapchat',
    131: 'Funny', 132: 'Celebrations/Events', 133: 'Tom Holland & Zendaya', 134: 'Voice Effects',
    135: 'Incoherent', 136: 'Incoherent', 137: 'Incoherent', 138: 'Chiropractor', 139: 'Sad/depressed',
    140: 'Minecraft Streamers', 141: 'Incoherent', 142: 'Animal Crossing', 143: 'Vibes', 144: 'Rocket League',
    145: 'K-Pop', 146: 'Cleaning', 147: 'Life Hacks', 148: 'Splatoon', 149: 'Incoherent', 150: 'Incoherent',
    151: 'Gossip Girl', 152: 'Twins', 153: 'Animals', 154: 'Funny', 155: 'Incoherent', 156: 'Autism',
    157: 'Piano', 158: 'Incoherent', 159: 'Incoherent', 160: 'Incoherent', 161: 'Singing',
    162: 'Star Wars: Battlefront II', 163: 'Metal Music', 164: 'Farming', 165: 'Weddings',
    166: 'Arab/Muslim', 167: 'Tourettes', 168: 'Hazbin Hotel', 169: 'One Direction', 170: 'Girls',
    171: 'Tattoos', 172: 'Greys Anatomy', 173: 'Memories', 174: 'Thai Boys Love shows',
    175: 'Toilet-Bound Hanako-kun', 176: 'Incoherent', 177: 'Football', 178: 'Violin',
    179: 'One Direction', 180: 'Incoherent', 181: 'Attack on Titan', 182: 'Incoherent',
    183: 'Percy Jackson', 184: 'Incoherent', 185: 'Friends (TV Show)', 186: 'New Year',
    187: 'Minecraft', 188: 'EDM', 189: 'Sleep', 190: 'Riverdale', 191: 'The Chronicles of Narnia',
    192: 'Cats', 193: 'Psychology', 194: 'Funny', 195: 'Glow-up', 196: 'My Hero Academia',
    197: 'My Hero Academia', 198: 'Christianity', 199: 'My Hero Academia', 200: 'Trucking',
    201: 'FIFA/EA', 202: 'My Hero Academia', 203: 'Belgium', 204: 'Insects', 205: 'Amazon',
    206: 'Spotify', 207: 'Vlogs', 208: 'Incoherent', 209: 'Fairy Tale', 210: 'Alcohol',
    211: 'Bridgerton', 212: 'Incoherent', 213: 'Criminal Minds', 214: 'Ateez', 215: 'Incoherent',
    216: 'Hamsters', 217: 'Teeth', 218: 'Plumbing', 219: 'Incoherent', 220: 'Snow/Ice/Winter',
    221: 'My Hero Academia', 222: 'Dance', 223: 'Dinosaurs', 224: '2024 US election', 225: 'Food', 226: 'Pregnancy',
    227: 'Thrifting',
}

In [19]:
# ── Outlier model labels ─────────────────────────────────────────────────────
outlier_topic_mapping = {
    -1: 'Outliers', 0: 'Football', 1: 'Taylor Swift', 2: 'Funny', 3: 'Genshin Impact', 4: 'Incoherent',
    5: 'K-Pop', 6: 'Motivational', 7: 'Cars', 8: 'Cats', 9: 'Incoherent', 10: 'Martial arts', 11: 'Eurovision',
    12: 'Bankzitters', 13: 'Game of Thrones', 14: 'Netherlands', 15: 'ASMR',
    16: 'Couples', 17: 'Karate Kid', 18: 'Dance', 19: 'The Hunger Games', 20: 'Feminism', 21: 'Billie Eilish', 22: 'Jujutsu Kaisen',
    23: 'Minecraft', 24: 'Dark Romance', 25: 'Making Money', 26: 'Formula 1', 27: 'Hockey', 28: 'Incoherent',
    29: 'Ateez', 30: 'Incoherent', 31: 'CapCut Editing', 32: 'Incoherent', 33: 'Geography', 34: 'Sped up music',
    35: 'Podcasts/radio', 36: 'Cosplay', 37: 'Manhwa', 38: 'Work', 39: 'Facts', 40: 'Pregnancy', 41: 'Football',
    42: 'Music', 43: 'Cooking', 44: 'Perfume', 45: 'Lego', 46: 'Hatsune Miku', 47: 'The Walking Dead', 48: 'Theme Parks',
    49: 'Poetry', 50: 'Advice', 51: 'Eating disorder recovery', 52: 'Scandinavia', 53: 'Incoherent', 54: 'Incoherent',
    55: 'Incoherent', 56: 'Brawl Stars', 57: 'Wealth', 58: 'Africa', 59: 'Twitch streamers', 60: 'Snow/Ice/Winter',
    61: 'Incoherent', 62: 'Grief', 63: 'Funny', 64: 'Music festivals', 65: 'Friends', 66: 'Tomorrow x Together',
    67: 'Demon Slayer', 68: 'France', 69: 'Positivity', 70: 'Fortnite', 71: 'Shopping', 72: 'Couples', 73: 'Gaming',
    74: 'The Sims', 75: 'Titanic (movie)', 76: 'Family Guy', 77: 'Incoherent', 78: 'Incoherent', 79: 'Girls',
    80: 'The Weeknd', 81: 'Justin Bieber', 82: 'NBA 2k', 83: 'Incoherent', 84: 'Farming', 85: 'Turkey',
    86: 'Construction', 87: 'Incoherent', 88: 'NFL', 89: 'Rainbow Six Siege', 90: 'Twice', 91: 'Heartbreak',
    92: 'Motivational', 93: 'Steven Universe', 94: 'Pirates of the Caribbean', 95: 'Audio', 96: 'Driving lessons',
    97: 'Singing', 98: 'South Asia', 99: 'History', 100: 'Total Drama Island', 101: 'Trauma', 102: 'Fanfiction',
    103: 'Bridgerton', 104: 'Food', 105: 'Love Island', 106: 'Prom', 107: 'Once Upon A Time', 108: 'Rappers',
    109: 'Ariana Grande', 110: 'Reddit stories', 111: 'Music', 112: 'Plants/gardening', 113: 'Call of Duty',
    114: 'Fitness', 115: 'Clash Royale', 116: 'Sturniolo triplets', 117: 'Quotes', 118: 'Greece', 119: 'Street interviews',
    120: 'Generations', 121: 'Incoherent', 122: 'Resin art', 123: 'Clothing shopping', 124: 'Italy', 125: 'Incoherent',
    126: 'Incoherent', 127: 'Latin America/Caribbean', 128: 'Incoherent', 129: 'Balkans', 130: 'Concerts', 131: 'Blue Lock',
    132: 'Incoherent', 133: 'Sam and Colby', 134: 'Studying', 135: 'Miraculous Ladybug', 136: 'Modelling (fashion)',
    137: 'Reddit stories', 138: 'Incoherent', 139: 'Memes', 140: 'Rappers', 141: 'My Hero Academia', 142: 'Incoherent',
    143: 'Enhypen', 144: 'Animals', 145: 'Enhypen', 146: 'Incoherent', 147: 'Gidle', 148: 'The Amazing Digital Circus',
    149: 'Pinterest aesthetic', 150: 'Vintage fashion', 151: 'Vlogs', 152: 'Driving lessons', 153: 'Taylor Swift',
    154: 'Fashion', 155: 'Body positivity', 156: 'Spongebob', 157: 'K2 zoekt K3', 158: 'Incoherent', 159: 'Incoherent',
    160: 'Disability', 161: 'Fashion', 162: 'Rabbits', 163: 'Chronic illness/diseases', 164: 'Van life', 165: 'Incoherent',
    166: 'Harry Styles', 167: 'Golf', 168: 'Fire/rescue', 169: 'Incoherent', 170: 'Languages', 171: 'Incoherent',
    172: '3D printing', 173: 'Incoherent', 174: 'The Summer I Turned Pretty', 175: 'Incoherent', 176: 'Incoherent',
    177: 'Swimsuits', 178: 'Incoherent', 179: 'Bungou Stray Dogs', 180: 'Cristiano Ronaldo', 181: 'Incoherent', 182: 'Flowers',
    183: 'Art', 184: 'Fortnite', 185: 'Feyenoord FC', 186: 'Retro music', 187: 'Incoherent', 188: 'Lawyers', 189: 'Drums',
    190: 'Incoherent', 191: 'Incoherent', 192: 'Incoherent', 193: 'Couples', 194: 'Incoherent', 195: 'Rappers',
    196: 'My little pony', 197: 'Olivia Rodrigo', 198: 'Glee', 199: 'Incoherent', 200: 'Body positivity',
    201: 'Incoherent', 202: 'Chronic illness/diseases', 203: 'The Voice', 204: 'Heartbreak', 205: 'Incoherent',
    206: 'Heartstopper', 207: 'How to Train Your Dragon', 208: 'Incoherent', 209: 'Science', 210: 'Incoherent',
    211: 'Love', 212: 'Incoherent', 213: 'Incoherent', 214: 'Brugklas', 215: 'Incoherent', 216: 'Incoherent',
    217: 'Cancer', 218: 'Incoherent', 219: 'Sad/depressed', 220: 'Ice skating', 221: 'Books', 222: 'FIFA/EA',
    223: 'Old money', 224: 'Outer Banks', 225: 'Minecraft', 226: 'Incoherent', 227: 'Incoherent', 228: 'Self-love',
    229: 'Incoherent', 230: 'One Piece', 231: 'Toxic relationships', 232: 'Lord of the Rings', 233: 'Incoherent',
    234: 'Sidemen', 235: 'Europe', 236: 'Wednesday Addams', 237: 'Incoherent', 238: 'Incoherent', 239: 'Monster High',
    240: 'Incoherent', 241: 'Incoherent', 242: 'Backpacking', 243: 'Incoherent', 244: 'Incoherent', 245: 'Sustainability',
    246: 'Nails', 247: 'Incoherent',
}

In [20]:
# Sanity-check: flag IDs in the mappings that aren't in the models
model_main_ids    = set(topic_model.get_topic_info()["Topic"])
model_outlier_ids = set(topic_model_outliers.get_topic_info()["Topic"])

extra_main    = set(main_topic_mapping)    - model_main_ids
extra_outlier = set(outlier_topic_mapping) - model_outlier_ids

if extra_main:    print(f"⚠️  Main mapping has IDs not in model:    {sorted(extra_main)}")
if extra_outlier: print(f"⚠️  Outlier mapping has IDs not in model: {sorted(extra_outlier)}")
if not extra_main and not extra_outlier:
    print("✅ All mapping IDs found in their respective models")

✅ All mapping IDs found in their respective models


In [ ]:
merged_topic_info_renamed = rename_merged_topic_labels(
    merged_topic_info=merged_topic_info,
    main_topic_mapping=main_topic_mapping,
    outlier_topic_mapping=outlier_topic_mapping,
    offset=topic_id_mapping["offset"],
)

✅ Renamed 477 topic labels
   Main mapping entries:    229
   Outlier mapping entries: 249


In [22]:
merged_topic_info_renamed.sample(10, random_state=42)

,Topic,Count,Name,Representation,Representative_Docs,Source,topic_label_original
468,430,1072,Chronic illness/diseases,"[psoriasis, endometriosis, adenomyosis, womens...",[eczematok psoriasis tsw eczemaawareness eczem...,Outlier Model,202_psoriasis_endometriosis_adenomyosis_womens...
33,13,33839,Christmas,"[christmas, happyholidays, gift, monclerbubble...","[christmas secretsanta, christmas santaslays, ...",Main Model,13_christmas_happyholidays_gift_monclerbubbleup
131,53,8676,Incoherent,"[greenscreenvideo, greenscreen, snapchat, goog...","[greenscreenvideo, greenscreenvideo, greenscre...",Main Model,53_greenscreenvideo_greenscreen_snapchat_googl...
72,49,13892,Dangan Ronpa,"[danganronpa, tokofukawa, junkoenoshima, dr3, ...","[danganronpa, danganronpa, 11037 danganronpa]",Main Model,49_danganronpa_tokofukawa_junkoenoshima_dr3
78,68,13414,Beach,"[beach, mermaid, ocean, surfing, surf, h2o, wa...",[ocean surf fall brushboard stack waves wsl su...,Main Model,68_beach_mermaid_ocean_surfing
113,89,9827,Sewing,"[sewing, thriftflip, crochet, sewingtutorial, ...","[thriftflip sewing, diy thriftflip sewing, diy...",Main Model,89_sewing_thriftflip_crochet_sewingtutorial
274,337,3642,Ariana Grande,"[arianagrande, arianator, ariana, eternalsunsh...","[arianagrande, arianagrande, arianagrande umg]",Outlier Model,109_arianagrande_arianator_ariana_eternalsunshine
185,269,5839,Football,"[goalkeeper, goalkeepertraining, keeper, gk, g...","[goalkeeper, futbol goalkeeper footy keeper go...",Outlier Model,41_goalkeeper_goalkeepertraining_keeper_gk
261,65,3921,My Hero Academia,"[hawks, keigotakami, dabihawks, dabi, hawksbnh...","[anime hawks mha keigotakami, myheroacademia h...",Main Model,65_hawks_keigotakami_dabihawks_dabi
9,8,73748,Hair,"[hair, hairstyle, hairtutorial, haircut, curly...","[hairtutorial hair, hair hairtutorial, emohair...",Main Model,8_hair_hairstyle_hairtutorial_haircut


## 8 · Add overarching categories (human coding)

With 476 initial topics, we deduced that some of them are closely related, e.g., referring to jobs, communities, media franchises etc. 

Those were manually labeled and put together, and the code below applies this step to be reflected in the data frame, i.e., adding an additional column indicated which parent topic each topic belongs to.

In [23]:
def add_parent_topics(
    merged_topic_info: pd.DataFrame,
    parent_topic_mapping: dict,
    offset: int,
) -> pd.DataFrame:
    """
    Add a 'parent_topic' column from a parent→[topic_ids] mapping.

    Parameters
    ----------
    parent_topic_mapping : dict with keys 'main' and 'outlier', each mapping
                           category_name -> list of topic_ids (original IDs,
                           before offset).
    """
    # Invert to topic_id -> parent for both models
    combined: dict[int, str] = {}
    for parent, ids in parent_topic_mapping.get("main", {}).items():
        for tid in ids:
            combined[tid] = parent
    for parent, ids in parent_topic_mapping.get("outlier", {}).items():
        for tid in ids:
            if tid != -1:
                combined[tid + offset] = parent

    df = merged_topic_info.copy()
    df["parent_topic"] = df["Topic"].map(combined).fillna("Uncategorized")
    df.loc[df["Topic"] == -1, "parent_topic"] = "Outlier"

    uncategorized = (df["parent_topic"] == "Uncategorized").sum()
    print(f"✅ Parent topics added")
    if uncategorized:
        print(f"⚠️  {uncategorized} topics left as 'Uncategorized' — check mapping")
    return df

In [24]:
parent_topic_mapping = {
    'main': {
        'Comedy': [15, 66, 87, 131, 154, 194, 73, 112, 129],
        'Food and drinks': [0, 116, 210, 225],
        'Hobbies': [1, 27, 222, 40, 41, 50, 63, 83, 89, 100, 114],
        'Animals': [3, 32, 54, 64, 113, 127, 192, 153, 204, 216, 223],
        'Reading': [60],
        'LGBTQ+': [5],
        'Sports': [7, 30, 177, 37, 47, 88, 102, 124],
        'Style/Appearance': [8, 10, 18, 35, 42, 92, 120, 125, 171, 195, 227],
        'Education': [9, 52],
        'Geography': [11, 39, 75, 76, 90, 91, 203],
        'Holidays, seasonal, and events': [12, 13, 59, 68, 71, 132, 165, 186, 220],
        'Gaming': [14, 25, 49, 98, 104, 105, 109, 118, 140, 142, 144, 148, 162, 187, 201],
        'Media franchises': [4, 17, 6, 16, 22, 38, 70, 151, 168, 172, 174, 183, 185, 190, 191, 211, 213],
        'Emotional and reflective content': [143, 173],
        'Romantic relationships': [24, 80],
        'Family': [26, 81, 152, 226],
        'Advice': [147],
        'K-pop': [31, 145, 45, 84, 96, 214],
        'Anime and manga': [29, 33, 48, 65, 101, 107, 108, 196, 197, 199, 202, 221, 43, 44, 175, 181, 209, 369],
        'Religion and spirituality': [34, 72, 166, 198],
        'COVID-19': [36, 74],
        'Friendship': [46, 170],
        'Companies/products': [51, 126, 130, 205],
        'Jobs': [57, 110, 122, 164, 200, 218],
        'Playing/producing music': [61, 95, 117, 157, 161, 178, 188],
        'TikTok video types': [67, 69, 86, 103, 134, 207],
        'Mental health': [79, 128, 139, 156, 167, 193],
        'Homemaking': [93, 146],
        'Politics and social issues': [97, 224],
        'Listening to music': [99, 123, 163, 169, 179, 206, 368, 423],
        'Celebrities/influencers': [106, 133],
        'Physical health': [138, 189, 217],
        'Incoherent': [2, 19, 20, 21, 23, 28, 53, 55, 56, 58, 62, 77, 78, 82, 85, 94, 111, 115,
                       119, 121, 135, 136, 137, 141, 149, 150, 155, 158, 159, 160, 176, 180, 182,
                       184, 208, 212, 215, 219],
    },
    'outlier': {
        'Comedy': [2, 63, 139],
        'Food and drinks': [43, 104],
        'Hobbies': [183, 18, 7, 36, 35, 45, 48, 71, 112, 122, 164, 172, 182],
        'Animals': [8, 144, 162],
        'Reading': [221, 24, 49, 102],
        'Sports': [114, 0, 41, 10, 26, 27, 88, 167, 185, 220],
        'Style/Appearance': [154, 161, 246, 44, 123, 149, 150, 177, 223],
        'Education': [39, 96, 152, 99, 134, 209],
        'Geography': [14, 170, 235, 33, 52, 58, 68, 85, 98, 118, 124, 127, 129, 242],
        'Holidays, seasonal, and events': [60, 11, 106],
        'Gaming': [73, 70, 184, 23, 225, 222, 3, 56, 74, 82, 89, 113, 115],
        'Media franchises': [103, 13, 17, 19, 47, 75, 76, 93, 94, 100, 105, 107, 135, 148, 156,
                             157, 174, 196, 198, 203, 206, 207, 214, 224, 232, 239, 236],
        'Emotional and reflective content': [15, 69, 117, 120],
        'Romantic relationships': [16, 72, 193, 91, 204, 211],
        'Family': [40],
        'Advice': [6, 92, 25, 50, 57],
        'K-pop': [5, 29, 66, 90, 143, 145, 147],
        'Anime and manga': [22, 37, 67, 131, 179, 230],
        'Friendship': [65, 79],
        'Jobs': [84, 38, 86, 136, 168, 188],
        'Playing/producing music': [97, 189],
        'TikTok video types': [151, 31, 110, 137, 119],
        'Mental health': [219, 51, 62, 101, 155, 200, 228, 231],
        'Politics and social issues': [20, 245],
        'Listening to music': [1, 153, 21, 34, 42, 46, 111, 64, 80, 95, 108, 109, 130, 166, 186, 197],
        'Celebrities/influencers': [12, 59, 81, 116, 133, 180, 234],
        'Physical health': [160, 163, 202, 217],
        'Incoherent': [4, 9, 28, 30, 32, 53, 54, 55, 61, 77, 78, 83, 87, 121, 125, 126, 128, 132,
                       138, 142, 146, 158, 159, 165, 169, 171, 173, 175, 176, 178, 181, 187, 190,
                       191, 192, 194, 199, 201, 205, 208, 210, 212, 213, 215, 216, 218, 226, 227,
                       229, 233, 237, 238, 240, 241, 243, 244, 247],
    },
}

In [25]:
merged_topic_info_with_category = (
    add_parent_topics(
        merged_topic_info=merged_topic_info_renamed,
        parent_topic_mapping=parent_topic_mapping,
        offset=topic_id_mapping["offset"],
    )
    .rename(columns={"parent_topic": "Category", "Name": "Topic Label"})
)

✅ Parent topics added


In [26]:
merged_topic_info_with_category.sample(5, random_state=42)

,Topic,Count,Topic Label,Representation,Representative_Docs,Source,topic_label_original,Category
468,430,1072,Chronic illness/diseases,"[psoriasis, endometriosis, adenomyosis, womens...",[eczematok psoriasis tsw eczemaawareness eczem...,Outlier Model,202_psoriasis_endometriosis_adenomyosis_womens...,Physical health
33,13,33839,Christmas,"[christmas, happyholidays, gift, monclerbubble...","[christmas secretsanta, christmas santaslays, ...",Main Model,13_christmas_happyholidays_gift_monclerbubbleup,"Holidays, seasonal, and events"
131,53,8676,Incoherent,"[greenscreenvideo, greenscreen, snapchat, goog...","[greenscreenvideo, greenscreenvideo, greenscre...",Main Model,53_greenscreenvideo_greenscreen_snapchat_googl...,Incoherent
72,49,13892,Dangan Ronpa,"[danganronpa, tokofukawa, junkoenoshima, dr3, ...","[danganronpa, danganronpa, 11037 danganronpa]",Main Model,49_danganronpa_tokofukawa_junkoenoshima_dr3,Gaming
78,68,13414,Beach,"[beach, mermaid, ocean, surfing, surf, h2o, wa...",[ocean surf fall brushboard stack waves wsl su...,Main Model,68_beach_mermaid_ocean_surfing,"Holidays, seasonal, and events"


In [27]:
# Aggregate counts
category_dist = (
    merged_topic_info_with_category
    .groupby("Category", as_index=False)
    .agg(Count=("Count", "sum"), N_topics=("Topic", "count"))
)

topic_dist = (
    merged_topic_info_with_category
    .groupby("Topic Label", as_index=False)
    .agg(Count=("Count", "sum"))
)

In [ ]:
# Save merged topic info
# merged_topic_info_with_category.to_excel(MERGED_TOPICS_CATEGORIES_PATH, index=False)

## 9 · Find fragmented / overlapping topics

Uses Jaccard overlap of top-N hashtags to flag:
- Topics in the **outlier model** that overlap with **main model** topics (redundancy)
- Topics within the **same model** that are suspiciously similar (fragmentation)

Results are passed to notebook 4 for deeper exploration.

In [28]:
def find_fragmented_topics(
    topic_model: BERTopic,
    topic_model_outliers: BERTopic,
    top_n: int = 20,
    similarity_threshold: float = 0.3,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Identify overlapping and fragmented topics across and within models.

    Returns
    -------
    cross_overlaps_df  : Main ↔ outlier model pairs above threshold.
    outlier_frags_df   : Pairs within the outlier model above threshold.
    main_overlaps_df   : Pairs within the main model above threshold.
    """

    def _top_words(model, topic_id):
        words = model.get_topic(topic_id)
        return set(w for w, _ in words[:top_n]) if words else set()

    def _jaccard_pairs(model_a, ids_a, model_b, ids_b, label_a="a", label_b="b"):
        rows = []
        same_model = model_a is model_b
        for i, id_a in enumerate(ids_a):
            wa = _top_words(model_a, id_a)
            if not wa:
                continue
            start = i + 1 if same_model else 0
            for id_b in (ids_b[start:] if same_model else ids_b):
                wb = _top_words(model_b, id_b)
                if not wb:
                    continue
                inter = wa & wb
                jaccard = len(inter) / len(wa | wb)
                if jaccard >= similarity_threshold:
                    rows.append({
                        label_a: id_a,
                        label_b: id_b,
                        "jaccard": round(jaccard, 3),
                        "shared_hashtags": inter,
                    })
        return pd.DataFrame(rows).sort_values("jaccard", ascending=False) if rows else pd.DataFrame()

    main_ids    = [t for t in topic_model.get_topic_info()["Topic"] if t != -1]
    outlier_ids = [t for t in topic_model_outliers.get_topic_info()["Topic"] if t != -1]

    print("=" * 60)
    print("MAIN ↔ OUTLIER OVERLAP")
    cross = _jaccard_pairs(topic_model, main_ids, topic_model_outliers, outlier_ids,
                           "main_topic_id", "outlier_topic_id")
    print(f"  Overlapping pairs: {len(cross)}")
    if not cross.empty:
        print(cross.head(10).to_string(index=False))

    print("\n" + "=" * 60)
    print("FRAGMENTATION WITHIN OUTLIER MODEL")
    out_frags = _jaccard_pairs(topic_model_outliers, outlier_ids, topic_model_outliers, outlier_ids,
                               "topic_a", "topic_b")
    print(f"  Fragmented pairs: {len(out_frags)}")
    if not out_frags.empty:
        print(out_frags.head(10).to_string(index=False))

    print("\n" + "=" * 60)
    print("FRAGMENTATION WITHIN MAIN MODEL")
    main_frags = _jaccard_pairs(topic_model, main_ids, topic_model, main_ids,
                                "topic_a", "topic_b")
    print(f"  Fragmented pairs: {len(main_frags)}")
    if not main_frags.empty:
        print(main_frags.head(10).to_string(index=False))

    return cross, out_frags, main_frags

In [29]:
cross_overlaps_df, outlier_frags_df, main_frags_df = find_fragmented_topics(
    topic_model=topic_model,
    topic_model_outliers=topic_model_outliers,
    top_n=20,
    similarity_threshold=0.3,
)

MAIN ↔ OUTLIER OVERLAP
  Overlapping pairs: 11
 main_topic_id  outlier_topic_id  jaccard                                                                shared_hashtags
            11                14    0.538         {nederland, nl, rotterdam, netherlands, dutch, dutchtiktok, amsterdam}
            48               141    0.538                         {deku, mha, todoroki, kirishima, bakugou, bnha, denki}
            33               141    0.538                {deku, mha, todoroki, kirishima, bakugou, myheroacademia, bnha}
           196               141    0.538             {deku, todoroki, kirishima, bakugou, myheroacademia, bnha, aizawa}
            10               161    0.429                      {outfitideas, ootd, fashionhacks, style, outfit, fashion}
           109                70    0.429 {fortniteclips, fortnitecreative, fortnite, fortnitememes, gaming, ogfortnite}
            27                18    0.429                    {ballet, dancing, dancemoms, dance, dancechal

## 10 · Link topics to videos

Create the final per-video DataFrame that maps each video ID to its topic label and category.  

This is the primary output consumed by notebook 4.

**The code below was used to attach the generated topic labels during this pipeline back to the original videos.**

**For anonymization, this is greyed out as IDs and possible links to the original videos had to be erased.**

In [30]:
def create_video_topic_mapping(
    docs_bert: list,
    merged_topics: np.ndarray,
    merged_topic_info: pd.DataFrame,
    video_ids: pd.Series,
) -> pd.DataFrame:
    """
    Produce a per-video DataFrame with topic label, category, and source model.
    """
    label_map    = dict(zip(merged_topic_info["Topic"], merged_topic_info["Topic Label"]))
    category_map = dict(zip(merged_topic_info["Topic"], merged_topic_info["Category"]))
    source_map   = dict(zip(merged_topic_info["Topic"], merged_topic_info["Source"]))

    df = pd.DataFrame({
        "video_id":     video_ids.values,
        "document":     docs_bert,
        "topic_id":     merged_topics,
        "topic_label":  [label_map.get(t,    "Outlier")       for t in merged_topics],
        "category":     [category_map.get(t, "Uncategorized") for t in merged_topics],
        "source_model": [source_map.get(t,   "Unknown")       for t in merged_topics],
    })

    print(f"✅ Video-topic mapping: {len(df):,} rows")
    print("\nBy source model:")
    print(df["source_model"].value_counts().to_string())
    print("\nBy category (top 10):")
    print(df["category"].value_counts().head(10).to_string())
    return df

In [31]:
video_topic_df = create_video_topic_mapping(
    docs_bert=docs_bert,
    merged_topics=merged_topics,
    merged_topic_info=merged_topic_info_with_category,
    video_ids=hashtag_df["id"],
)

✅ Video-topic mapping: 5,332,704 rows

By source model:
source_model
Main Model       4360743
Outlier Model     971961

By category (top 10):
category
Incoherent                        603440
Media franchises                  592670
Outlier                           569288
Sports                            377497
Style/Appearance                  327443
Hobbies                           308989
Gaming                            256790
Geography                         208408
Holidays, seasonal, and events    186201
Animals                           175033


In [ ]:
video_topic_df.head()

## 11 · Save outputs

In [ ]:
# Uncomment each line when ready to write

# np.save(MERGED_TOPICS_PATH, merged_topics)

# merged_topic_info.to_csv(MERGED_INFO_CSV_PATH, index=False)

# with pd.ExcelWriter(CATEGORY_COUNT_PATH, engine='openpyxl') as w:
#     category_dist.to_excel(w, sheet_name='Category counts', index=False)

# with pd.ExcelWriter(TOPIC_COUNT_PATH, engine='openpyxl') as w:
#     topic_dist.to_excel(w, sheet_name='Topic counts', index=False)

# video_topic_df.to_pickle(DOCS_VIDEO_PATH)

---
## AI disclosure

AI tools were used to assist with developing, labelling, and debugging code, and with formatting Markdown cells.

Tools used: [Cursor](https://cursor.com/agents) · [Claude](https://claude.ai/) · [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs produced by AI tools and accept full accountability for their accuracy and validity.

*XXX*